# 6.20 — Learning Rate Schedules

A learning-rate schedule chooses the step size $\eta_t$ used by gradient descent at each training step. In this lesson, you will build constant, warmup, step-decay, cosine-decay, and one-cycle schedules from scratch in NumPy, then inspect how each schedule changes the actual parameter movement on a tiny optimization problem.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build learning-rate schedules one idea at a time. Run each cell in order and read the printed intermediate values — every schedule is just arithmetic on the step index, and every update is shown. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, schedule formulas, and tiny optimization loops.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any randomized toy data.

### 1. The learning rate is the size of a gradient step

Gradient descent moves a parameter by $\theta_{t+1}=\theta_t-\eta_t g_t$. The gradient $g_t$ points uphill for the loss, so the minus sign moves downhill. The learning rate $\eta_t$ is not the direction; it is the distance multiplier. A schedule matters because the same gradient can produce a cautious nudge or an overshooting jump depending only on $\eta_t$.

In [ ]:
theta_w = 2.0                         # one scalar parameter before the update.
grad_w = 2.1                          # current derivative dL/dtheta.
eta_w = 0.05                          # learning rate for this step.
move_w = eta_w * grad_w               # amount subtracted from theta.
theta_next_w = theta_w - move_w        # gradient descent update.
print("move:", round(move_w, 3))
print("theta next:", round(theta_next_w, 3))
assert round(move_w, 3) == 0.105
assert round(theta_next_w, 3) == 1.895

▶ What you'll see: the gradient says which way to move, and the learning rate scales that move to 0.105.

In [ ]:
eta_grid_w = np.array([0.01, 0.05, 0.20])        # compare small, moderate, and large rates.
next_grid_w = theta_w - eta_grid_w * grad_w      # same gradient, different step sizes.
print("next theta values:", np.round(next_grid_w, 3))
plt.figure(figsize=(4.4, 3))
plt.bar(["0.01", "0.05", "0.20"], theta_w - next_grid_w, color="teal")
plt.title("1: same gradient, different movement")
plt.xlabel("learning rate η")
plt.ylabel("subtracted amount ηg")
plt.show()

▶ What you'll see: movement grows linearly with the learning rate; the direction is unchanged.

*Why it's done this way:* the derivative is measured in loss-change per parameter-change, so multiplying by $\eta_t$ converts slope information into an actual parameter displacement. Scheduling $\eta_t$ is therefore scheduling how much trust we place in each noisy gradient estimate.

### 2. Warmup: start small while signals stabilize

Warmup increases the rate gradually from a tiny value to a target maximum. Early gradients can be large or poorly calibrated because weights, activations, and optimizer statistics are still settling. A linear warmup says: spend the first few steps learning the scale of the problem before taking full-size steps.

In [ ]:
steps_w = np.arange(12)                         # training step indices.
warmup_steps_w = 5                              # number of steps spent ramping up.
eta_max_w = 0.10                                # target learning rate after warmup.
warmup_w = eta_max_w * np.minimum(1.0, (steps_w + 1) / warmup_steps_w)
print("warmup rates:", np.round(warmup_w, 3))
assert np.allclose(np.round(warmup_w[:5], 3), [0.02, 0.04, 0.06, 0.08, 0.10])

▶ What you'll see: the schedule climbs 0.02, 0.04, 0.06, 0.08, 0.10, then stays at the target.

In [ ]:
grad0_w = 3.0                                  # pretend the early gradient is large.
updates_w = warmup_w[:6] * grad0_w             # actual movement caused by the schedule.
print("first six update sizes:", np.round(updates_w, 3))
plt.figure(figsize=(4.6, 3))
plt.plot(steps_w, warmup_w, marker="o", color="seagreen")
plt.title("2: linear warmup schedule")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: the plotted ramp prevents the earliest updates from being as large as later updates.

*Why it's done this way:* a linear ramp is the simplest interpolation between “almost do not move” and “use the intended rate.” The formula $\eta_t=\eta_{max}(t+1)/T_w$ keeps the increase predictable, so every early update is a controlled fraction of the full update.

### 3. Step decay: drop the rate when progress plateaus

Step decay keeps a rate constant for a while, then multiplies it by a factor such as 0.1 at chosen milestones. The idea is coarse but practical: use decisive movement early, then reduce the step size so the optimizer can settle into a narrower part of the loss surface.

In [ ]:
steps3_w = np.arange(16)                         # training steps to display.
eta0_w = 0.20                                    # initial learning rate.
drop_every_w = 5                                 # milestone interval.
gamma_w = 0.5                                    # multiplicative drop factor.
step_decay_w = eta0_w * gamma_w ** (steps3_w // drop_every_w)
print("step-decay rates:", np.round(step_decay_w, 3))
assert np.allclose(np.round(step_decay_w[[0, 5, 10, 15]], 3), [0.20, 0.10, 0.05, 0.025])

▶ What you'll see: the rate stays flat, drops at step 5, drops again at step 10, and drops again at step 15.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.step(steps3_w, step_decay_w, where="post", color="darkorange")
plt.scatter(steps3_w, step_decay_w, color="darkorange")
plt.title("3: step decay spends rate in plateaus")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: a staircase shape; the discontinuities are intentional milestone decisions.

*Why it's done this way:* multiplying by $\gamma$ changes the update scale without changing the gradient formula. Large plateaus help escape broad high-loss regions; later smaller plateaus make oscillation around a minimum less likely.

### 4. Cosine decay: settle smoothly instead of jumping

Cosine decay lowers the rate continuously from $\eta_{max}$ to $\eta_{min}$:
$$\eta_t=\eta_{min}+\frac12(\eta_{max}-\eta_{min})(1+\cos(\pi t/T)).$$
The cosine starts with a gentle slope, falls fastest in the middle, and ends gently. That shape avoids sudden changes while still spending most of training moving from bold steps to careful steps.

In [ ]:
T_w = 20                                      # total schedule length.
t_cos_w = np.arange(T_w + 1)                  # include both endpoints 0 and T.
eta_min_w = 0.01                              # final floor.
eta_max2_w = 0.10                             # initial ceiling.
cosine_w = eta_min_w + 0.5 * (eta_max2_w - eta_min_w) * (1 + np.cos(np.pi * t_cos_w / T_w))
print("start/middle/end:", np.round([cosine_w[0], cosine_w[10], cosine_w[-1]], 3))
assert np.allclose(np.round([cosine_w[0], cosine_w[10], cosine_w[-1]], 3), [0.10, 0.055, 0.01])

▶ What you'll see: the schedule begins at 0.100, reaches the midpoint 0.055 halfway through, and ends at 0.010.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.plot(t_cos_w, cosine_w, marker="o", color="purple")
plt.title("4: cosine decay from η_max to η_min")
plt.xlabel("step t")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: a smooth falling curve with no sharp milestone jumps.

*Why it's done this way:* the term $(1+\cos(\pi t/T))/2$ is exactly 1 at $t=0$ and 0 at $t=T$, so it acts like a smooth interpolation weight between maximum and minimum learning rates.

### 5. One-cycle: explore upward, then anneal downward

A one-cycle schedule first increases the learning rate, then decreases it to a very small value. The rising half can help the optimizer move out of narrow or poorly conditioned regions; the falling half then anneals the updates so the final parameters settle. We can build it by concatenating two linear pieces.

In [ ]:
up_steps_w = 5                                  # length of the rising phase.
down_steps_w = 10                               # length of the falling phase.
eta_low_w = 0.02                                # beginning rate.
eta_peak_w = 0.12                               # exploratory peak.
eta_final_w = 0.005                             # final annealed rate.
up_w = np.linspace(eta_low_w, eta_peak_w, up_steps_w, endpoint=False)
down_w = np.linspace(eta_peak_w, eta_final_w, down_steps_w)
one_cycle_w = np.concatenate([up_w, down_w])
print("one-cycle endpoints:", np.round([one_cycle_w[0], one_cycle_w[4], one_cycle_w[5], one_cycle_w[-1]], 3))
assert np.allclose(np.round([one_cycle_w[0], one_cycle_w[5], one_cycle_w[-1]], 3), [0.02, 0.12, 0.005])

▶ What you'll see: the schedule starts low, reaches a peak, then finishes below where it started.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.plot(np.arange(len(one_cycle_w)), one_cycle_w, marker="o", color="crimson")
plt.title("5: one-cycle learning rate")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: an up-then-down triangle-like curve, with the longest phase spent annealing.

*Why it's done this way:* one-cycle separates “search” from “settle.” The peak deliberately allows larger movement after warmup, while the final tiny rate reduces the risk that noisy minibatch gradients keep shaking the solution.

### 6. Schedules change optimization trajectories

To see schedules as training behavior rather than just curves, optimize $L(\theta)=(\theta-3)^2$. Its gradient is $2(\theta-3)$. A larger learning rate moves faster toward 3, but if it is too large it can cross the minimum repeatedly. Scheduled rates let us move aggressively early and carefully later.

In [ ]:
def grad_loss_w(theta):                         # derivative of (theta - 3)^2.
    return 2 * (theta - 3.0)

n_steps6_w = 30
constant_sched_w = np.full(n_steps6_w, 0.08)
cos_sched_w = 0.01 + 0.5 * (0.16 - 0.01) * (1 + np.cos(np.pi * np.arange(n_steps6_w) / (n_steps6_w - 1)))
print("constant first/last:", constant_sched_w[0], constant_sched_w[-1])
print("cosine first/last:", round(cos_sched_w[0], 3), round(cos_sched_w[-1], 3))
assert round(cos_sched_w[0], 3) == 0.16 and round(cos_sched_w[-1], 3) == 0.01

▶ What you'll see: the cosine schedule begins twice as high as the constant rate and ends much lower.

In [ ]:
def run_schedule_w(schedule):
    theta_hist = [0.0]
    for eta in schedule:
        theta_hist.append(theta_hist[-1] - eta * grad_loss_w(theta_hist[-1]))
    return np.array(theta_hist)

theta_const_w = run_schedule_w(constant_sched_w)
theta_cos_w = run_schedule_w(cos_sched_w)
print("final theta constant/cosine:", round(theta_const_w[-1], 3), round(theta_cos_w[-1], 3))
assert abs(theta_const_w[-1] - 3.0) < 0.02
assert abs(theta_cos_w[-1] - 3.0) < 0.02

▶ What you'll see: both trajectories end near the minimum, but they get there with different step sizes over time.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(theta_const_w, label="constant η=0.08", color="gray")
plt.plot(theta_cos_w, label="cosine 0.16→0.01", color="purple")
plt.axhline(3.0, color="black", linestyle="--", label="minimum θ=3")
plt.title("6: schedules create different paths")
plt.xlabel("step")
plt.ylabel("θ")
plt.legend()
plt.show()

▶ What you'll see: the cosine schedule moves quickly at first, then flattens as the rate decays.

*Why it's done this way:* optimization is repeated local approximation. Early in training we are far from the solution, so larger steps are useful; near the solution, smaller steps reduce bouncing around the minimum.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Each tiny example isolates one
> learning-rate schedule mechanic from the walkthrough, prints the intermediate values, draws one
> picture, and ends with an `assert` that pins the calculation.

### ✍️ Toy 1 · Learning rate scales a gradient step

The same gradient direction becomes a concrete movement only after multiplying by the learning rate.

In [ ]:
import numpy as np                              # arrays and gradient updates for this toy.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t1_theta = np.array([2.0, 1.0, -1.0, 0.0, 3.0, -2.0])   # six parameters.
t1_grad = np.array([2.1, -1.0, 0.5, 3.0, -2.0, 1.5])    # six gradients.
t1_eta = 0.05                                   # learning rate.
t1_move = t1_eta * t1_grad                      # movement size                     # -> [0.105, -0.05, 0.025, 0.15, -0.1, 0.075]
t1_next = t1_theta - t1_move                    # next parameters                   # -> [1.895, 1.05, -1.025, -0.15, 3.1, -2.075]
print("theta:", t1_theta.tolist())             # -> [2.0, 1.0, -1.0, 0.0, 3.0, -2.0]
print("gradient:", t1_grad.tolist())           # -> [2.1, -1.0, 0.5, 3.0, -2.0, 1.5]
print("eta*gradient:", np.round(t1_move, 3).tolist())  # -> [0.105, -0.05, 0.025, 0.15, -0.1, 0.075]
print("next theta:", np.round(t1_next, 3).tolist())    # -> [1.895, 1.05, -1.025, -0.15, 3.1, -2.075]
assert round(float(t1_next[0]), 3) == 1.895

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(t1_move.size), t1_move, color="teal")
plt.axhline(0, color="black", linewidth=0.7)
plt.xlabel("parameter")
plt.ylabel("subtracted amount")
plt.title("Toy 1 · eta scales the step")
plt.show()

▶ What you'll see: larger gradient coordinates move farther, scaled by the same `η=0.05`.

### ✍️ Toy 2 · Warmup ramps early rates linearly

Warmup starts with small updates and reaches the target rate over a fixed number of steps.

In [ ]:
import numpy as np                              # arrays and warmup formula for this toy.

t2_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t2_steps = np.arange(8)                         # eight training steps.
t2_warmup_steps = 4                             # ramp length.
t2_eta_max = 0.10                               # target learning rate.
t2_rates = t2_eta_max * np.minimum(1.0, (t2_steps + 1) / t2_warmup_steps)  # warmup rates.
t2_grad = 3.0                                   # fixed early gradient magnitude.
t2_updates = t2_rates * t2_grad                 # actual update sizes               # -> [0.075, 0.15, 0.225, 0.3, 0.3, 0.3, 0.3, 0.3]
print("steps:", t2_steps.tolist())             # -> [0, 1, 2, 3, 4, 5, 6, 7]
print("warmup rates:", np.round(t2_rates, 3).tolist())   # -> [0.025, 0.05, 0.075, 0.1, 0.1, 0.1, 0.1, 0.1]
print("update sizes:", np.round(t2_updates, 3).tolist()) # -> [0.075, 0.15, 0.225, 0.3, 0.3, 0.3, 0.3, 0.3]
assert np.allclose(np.round(t2_rates[:4], 3), [0.025, 0.05, 0.075, 0.1])

plt.figure(figsize=(4.8, 2.8))
plt.plot(t2_steps, t2_rates, marker="o", color="seagreen")
plt.xlabel("step")
plt.ylabel("eta_t")
plt.title("Toy 2 · warmup reaches the target")
plt.show()

▶ What you'll see: the first four points climb evenly, then the schedule stays flat.

### ✍️ Toy 3 · Step decay drops at milestones

Step decay keeps a plateau, then multiplies the rate by a fixed factor at each milestone.

In [ ]:
import numpy as np                              # arrays and integer milestone arithmetic for this toy.

t3_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t3_steps = np.arange(12)                        # twelve training steps.
t3_eta0 = 0.20                                  # initial rate.
t3_drop_every = 4                               # drop every four steps.
t3_gamma = 0.5                                  # half the rate each drop.
t3_stage = t3_steps // t3_drop_every            # plateau index                     # -> [0,0,0,0,1,1,1,1,2,2,2,2]
t3_rates = t3_eta0 * t3_gamma ** t3_stage       # step-decay rates                  # -> [0.2, ..., 0.05]
print("steps:", t3_steps.tolist())             # -> [0, 1, ..., 11]
print("plateau index:", t3_stage.tolist())     # -> [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]
print("step-decay rates:", np.round(t3_rates, 3).tolist())  # -> [0.2, 0.2, 0.2, 0.2, 0.1, 0.1, 0.1, 0.1, 0.05, 0.05, 0.05, 0.05]
assert np.allclose(np.round(t3_rates[[0, 4, 8]], 3), [0.2, 0.1, 0.05])

plt.figure(figsize=(4.8, 2.8))
plt.step(t3_steps, t3_rates, where="post", color="darkorange")
plt.scatter(t3_steps, t3_rates, color="darkorange")
plt.xlabel("step")
plt.ylabel("eta_t")
plt.title("Toy 3 · milestone staircase")
plt.show()

▶ What you'll see: the rate forms three flat plateaus, each half the previous one.

### ✍️ Toy 4 · Cosine decay interpolates smoothly

Cosine decay moves from a maximum to a minimum without sudden jumps.

In [ ]:
import numpy as np                              # arrays and cosine formula for this toy.

t4_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t4_T = 10                                       # final step index.
t4_t = np.arange(t4_T + 1)                      # eleven schedule points.
t4_eta_min = 0.01                               # final floor.
t4_eta_max = 0.11                               # starting ceiling.
t4_weight = 0.5 * (1 + np.cos(np.pi * t4_t / t4_T))     # smooth interpolation weight.
t4_rates = t4_eta_min + (t4_eta_max - t4_eta_min) * t4_weight  # cosine rates.
print("t:", t4_t.tolist())                     # -> [0, 1, ..., 10]
print("cosine weights:", np.round(t4_weight, 3).tolist())  # -> [1.0, 0.976, ..., 0.0]
print("cosine rates:", np.round(t4_rates, 3).tolist())     # -> [0.11, 0.108, ..., 0.01]
print("start/middle/end:", np.round([t4_rates[0], t4_rates[5], t4_rates[-1]], 3).tolist())  # -> [0.11, 0.06, 0.01]
assert np.allclose(np.round([t4_rates[0], t4_rates[5], t4_rates[-1]], 3), [0.11, 0.06, 0.01])

plt.figure(figsize=(4.8, 2.8))
plt.plot(t4_t, t4_rates, marker="o", color="purple")
plt.xlabel("step")
plt.ylabel("eta_t")
plt.title("Toy 4 · cosine falls smoothly")
plt.show()

▶ What you'll see: the curve starts high, passes through the midpoint rate, and ends at the floor.

### ✍️ Toy 5 · One-cycle rises, then anneals lower

One-cycle schedules concatenate a short rising phase with a longer falling phase.

In [ ]:
import numpy as np                              # arrays and concatenated linear schedules for this toy.

t5_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t5_up_steps = 4                                 # length of rising phase.
t5_down_steps = 6                               # length of falling phase.
t5_low = 0.02                                   # starting rate.
t5_peak = 0.12                                  # exploratory peak.
t5_final = 0.005                                # final annealed rate.
t5_up = np.linspace(t5_low, t5_peak, t5_up_steps, endpoint=False)  # rising rates # -> [0.02, 0.045, 0.07, 0.095]
t5_down = np.linspace(t5_peak, t5_final, t5_down_steps)            # falling rates # -> [0.12, 0.097, 0.074, 0.051, 0.028, 0.005]
t5_rates = np.concatenate([t5_up, t5_down])     # one-cycle schedule                # -> [0.02, ..., 0.005]
print("up phase:", np.round(t5_up, 3).tolist())        # -> [0.02, 0.045, 0.07, 0.095]
print("down phase:", np.round(t5_down, 3).tolist())    # -> [0.12, 0.097, 0.074, 0.051, 0.028, 0.005]
print("one-cycle rates:", np.round(t5_rates, 3).tolist())  # -> [0.02, 0.045, 0.07, 0.095, 0.12, 0.097, 0.074, 0.051, 0.028, 0.005]
assert np.allclose(np.round([t5_rates[0], t5_rates[4], t5_rates[-1]], 3), [0.02, 0.12, 0.005])

plt.figure(figsize=(4.8, 2.8))
plt.plot(np.arange(t5_rates.size), t5_rates, marker="o", color="crimson")
plt.xlabel("step")
plt.ylabel("eta_t")
plt.title("Toy 5 · one-cycle search then settle")
plt.show()

▶ What you'll see: the schedule climbs to a peak and then anneals below its starting rate.

### ✍️ Toy 6 · Schedules create different optimization paths

On the same quadratic loss, two schedules produce different parameter histories because they scale every update differently.

In [ ]:
import numpy as np                              # arrays and a tiny optimization loop for this toy.

t6_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t6_steps = 12                                   # twelve updates.
t6_constant = np.full(t6_steps, 0.08)           # constant schedule.
t6_cosine = 0.01 + 0.5 * (0.16 - 0.01) * (1 + np.cos(np.pi * np.arange(t6_steps) / (t6_steps - 1)))  # cosine schedule.

def t6_grad(theta):
    return 2 * (theta - 3.0)

def t6_run(schedule):
    t6_hist = [0.0]
    for t6_eta in schedule:
        t6_g = t6_grad(t6_hist[-1])
        t6_next = t6_hist[-1] - t6_eta * t6_g
        t6_hist.append(t6_next)
    return np.array(t6_hist)

t6_const_hist = t6_run(t6_constant)             # trajectory under constant rate    # -> final 2.63
t6_cos_hist = t6_run(t6_cosine)                 # trajectory under cosine rate      # -> final 2.712
t6_const_loss = (t6_const_hist - 3.0) ** 2      # constant losses.
t6_cos_loss = (t6_cos_hist - 3.0) ** 2          # cosine losses.
print("constant rates first/last:", round(float(t6_constant[0]), 3), round(float(t6_constant[-1]), 3))  # -> 0.08 0.08
print("cosine rates first/last:", round(float(t6_cosine[0]), 3), round(float(t6_cosine[-1]), 3))        # -> 0.16 0.01
print("constant theta history:", np.round(t6_const_hist, 3).tolist())  # -> [0.0, 0.48, ..., 2.63]
print("cosine theta history:", np.round(t6_cos_hist, 3).tolist())      # -> [0.0, 0.96, ..., 2.712]
print("final losses:", round(float(t6_const_loss[-1]), 3), round(float(t6_cos_loss[-1]), 3))            # -> 0.137 0.083
assert t6_const_loss[-1] < t6_const_loss[0]
assert t6_cos_loss[-1] < t6_cos_loss[0]

plt.figure(figsize=(4.8, 2.8))
plt.plot(t6_const_hist, marker="o", color="gray", label="constant")
plt.plot(t6_cos_hist, marker="s", color="purple", label="cosine")
plt.axhline(3.0, color="black", linestyle="--", label="minimum")
plt.xlabel("step")
plt.ylabel("theta")
plt.title("Toy 6 · schedules change the path")
plt.legend()
plt.show()

▶ What you'll see: the cosine path jumps faster at first, then flattens as its rate decays.


## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 🟢 Basics (warm-up)

### Basic 1 — Apply one gradient step

**Goal.** Use $\theta_{t+1}=\theta_t-\eta g$ once, because every schedule ultimately changes this one multiplication.

In [ ]:
theta_b1 = 2.0
grad_b1 = 2.1
eta_b1 = 0.05
next_b1 = theta_b1 - eta_b1 * grad_b1
print("next theta:", round(next_b1, 3))
assert round(next_b1, 3) == 1.895

In [ ]:
center_b1 = theta_b1 - grad_b1 / 2
grid_b1 = np.linspace(1.6, 2.2, 100)
loss_b1 = (grid_b1 - center_b1) ** 2
plt.figure(figsize=(4, 3))
plt.plot(grid_b1, loss_b1, color="slateblue")
plt.scatter([theta_b1, next_b1], [(theta_b1 - center_b1) ** 2, (next_b1 - center_b1) ** 2],
            color=["crimson", "seagreen"], zorder=3)
plt.annotate("before", (theta_b1, (theta_b1 - center_b1) ** 2), xytext=(5, 5), textcoords="offset points")
plt.annotate("after", (next_b1, (next_b1 - center_b1) ** 2), xytext=(5, -12), textcoords="offset points")
plt.title("Basic 1: one step on a 1D loss")
plt.xlabel("theta")
plt.ylabel("toy loss")
plt.show()

▶ What you'll see: the single update moves theta leftward and slightly down the toy quadratic.

▶ What you'll see: the parameter moves from 2.000 to 1.895.

👀 Takeaway: the learning rate scales the gradient before subtraction.

### Basic 2 — Compare update sizes

**Goal.** Keep the gradient fixed and vary the learning rate, because this isolates what the schedule controls.

In [ ]:
grad_b2 = 2.0
etas_b2 = np.array([0.01, 0.05, 0.10, 0.20])
updates_b2 = etas_b2 * grad_b2
print("updates:", np.round(updates_b2, 3))
assert np.allclose(updates_b2, [0.02, 0.10, 0.20, 0.40])
plt.figure(figsize=(4, 3))
plt.bar([str(x) for x in etas_b2], updates_b2, color="teal")
plt.title("Basic 2: update = ηg")
plt.xlabel("η")
plt.ylabel("update size")
plt.show()

▶ What you'll see: doubling the learning rate doubles the update.

👀 Takeaway: schedules are multiplicative controls on step length.

### Basic 3 — Build a constant schedule

**Goal.** Make an array of identical learning rates, because constant-rate training is the baseline schedule.

In [ ]:
steps_b3 = np.arange(8)
eta_b3 = 0.05
sched_b3 = np.full_like(steps_b3, eta_b3, dtype=float)
print("constant schedule:", sched_b3)
assert np.allclose(sched_b3, 0.05)
plt.figure(figsize=(4, 3))
plt.plot(steps_b3, sched_b3, marker="o", color="gray")
plt.ylim(0, 0.08)
plt.title("Basic 3: constant η")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: a flat line at 0.05.

👀 Takeaway: a constant schedule trusts every step equally.

### Basic 4 — Build linear warmup

**Goal.** Ramp from zero toward a target rate, because early updates are often the most fragile.

In [ ]:
steps_b4 = np.arange(6)
warmup_steps_b4 = 5
eta_max_b4 = 0.10
sched_b4 = eta_max_b4 * np.minimum(1.0, (steps_b4 + 1) / warmup_steps_b4)
print("warmup:", np.round(sched_b4, 3))
assert np.allclose(np.round(sched_b4, 3), [0.02, 0.04, 0.06, 0.08, 0.10, 0.10])
plt.figure(figsize=(4, 3))
plt.plot(steps_b4, sched_b4, marker="o", color="seagreen")
plt.title("Basic 4: linear warmup")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: a straight ramp that reaches 0.10 and then stays there.

👀 Takeaway: warmup makes early movement a fraction of the full learning rate.

### Basic 5 — Build step decay

**Goal.** Drop the learning rate at fixed milestones, because smaller late steps help settling.

In [ ]:
steps_b5 = np.arange(12)
eta0_b5 = 0.20
gamma_b5 = 0.5
drop_every_b5 = 4
sched_b5 = eta0_b5 * gamma_b5 ** (steps_b5 // drop_every_b5)
print("step decay:", np.round(sched_b5, 3))
assert np.allclose(np.round(sched_b5[[0, 4, 8]], 3), [0.20, 0.10, 0.05])
plt.figure(figsize=(4, 3))
plt.step(steps_b5, sched_b5, where="post", color="darkorange")
plt.title("Basic 5: milestone drops")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: a staircase from 0.20 to 0.10 to 0.05.

👀 Takeaway: step decay changes the trust level abruptly at chosen milestones.

### Basic 6 — Build cosine decay

**Goal.** Compute the core cosine formula, because it is a smooth high-to-low schedule.

In [ ]:
T_b6 = 10
t_b6 = np.arange(T_b6 + 1)
eta_min_b6 = 0.01
eta_max_b6 = 0.10
sched_b6 = eta_min_b6 + 0.5 * (eta_max_b6 - eta_min_b6) * (1 + np.cos(np.pi * t_b6 / T_b6))
print("cosine endpoints:", round(sched_b6[0], 3), round(sched_b6[-1], 3))
assert round(sched_b6[0], 3) == 0.10 and round(sched_b6[-1], 3) == 0.01
plt.figure(figsize=(4, 3))
plt.plot(t_b6, sched_b6, marker="o", color="purple")
plt.title("Basic 6: cosine decay")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: the curve starts at the maximum and lands exactly on the minimum.

👀 Takeaway: cosine decay is smooth interpolation from bold to careful updates.

### Basic 7 — Clip a schedule floor

**Goal.** Keep the rate from going below a floor, because some training runs should continue learning slowly instead of stopping.

In [ ]:
raw_b7 = np.linspace(0.10, -0.02, 7)
floor_b7 = 0.01
clipped_b7 = np.maximum(raw_b7, floor_b7)
print("raw:", np.round(raw_b7, 3))
print("floored:", np.round(clipped_b7, 3))
assert clipped_b7[-1] == 0.01
plt.figure(figsize=(4, 3))
plt.plot(raw_b7, marker="o", label="raw")
plt.plot(clipped_b7, marker="s", label="floored")
plt.title("Basic 7: learning-rate floor")
plt.legend()
plt.show()

▶ What you'll see: the floored schedule stops descending once it reaches 0.01.

👀 Takeaway: a floor preserves tiny but nonzero movement.

### Basic 8 — Compute cumulative learning-rate budget

**Goal.** Sum rates across steps, because total scheduled movement depends on both rate height and duration.

In [ ]:
sched_b8 = np.array([0.10, 0.10, 0.05, 0.05, 0.01])
budget_b8 = np.cumsum(sched_b8)
print("cumulative budget:", np.round(budget_b8, 3))
assert round(budget_b8[-1], 3) == 0.31
plt.figure(figsize=(4, 3))
plt.plot(budget_b8, marker="o", color="navy")
plt.title("Basic 8: cumulative η budget")
plt.xlabel("step")
plt.ylabel("sum of η so far")
plt.show()

▶ What you'll see: the budget rises fastest during high-rate steps.

👀 Takeaway: schedules allocate a finite amount of movement over time.

### Basic 9 — Apply a schedule to fixed gradients

**Goal.** Convert a rate schedule into actual update magnitudes, because the schedule affects parameters only through updates.

In [ ]:
grads_b9 = np.array([3.0, 2.0, 1.0, 0.5])
sched_b9 = np.array([0.02, 0.04, 0.08, 0.08])
updates_b9 = sched_b9 * grads_b9
print("updates:", np.round(updates_b9, 3))
assert np.allclose(updates_b9, [0.06, 0.08, 0.08, 0.04])
plt.figure(figsize=(4, 3))
plt.bar(np.arange(4), updates_b9, color="crimson")
plt.title("Basic 9: η_t times gradient")
plt.xlabel("step")
plt.ylabel("update magnitude")
plt.show()

▶ What you'll see: a larger rate can offset a smaller gradient, so update size is their product.

👀 Takeaway: the schedule and gradient jointly determine movement.

### Basic 10 — Plot schedule and loss together

**Goal.** Put a learning-rate curve beside a toy loss curve, because the schedule is meaningful only through training progress.

In [ ]:
steps_b10 = np.arange(20)
sched_b10 = 0.01 + 0.5 * (0.10 - 0.01) * (1 + np.cos(np.pi * steps_b10 / 19))
loss_b10 = np.exp(-0.18 * steps_b10) + 0.03 * sched_b10 / sched_b10.max()
print("first/last loss:", round(loss_b10[0], 3), round(loss_b10[-1], 3))
assert loss_b10[-1] < loss_b10[0]
plt.figure(figsize=(5, 3))
plt.plot(steps_b10, sched_b10 / sched_b10.max(), label="scaled η", color="purple")
plt.plot(steps_b10, loss_b10 / loss_b10.max(), label="scaled loss", color="teal")
plt.title("Basic 10: rate and progress")
plt.xlabel("step")
plt.legend()
plt.show()

▶ What you'll see: the rate decays while the toy loss decreases.

👀 Takeaway: schedules are judged by whether they help the loss decrease reliably.

## 🟡 Easy

### Easy 1 — Train a quadratic with constant and decayed rates

**Goal.** Optimize $L(\theta)=(\theta-3)^2$ with two schedules, because a schedule changes the trajectory even on a one-parameter problem.

In [ ]:
steps_e1 = 35
eta_const_e1 = np.full(steps_e1, 0.08)
eta_decay_e1 = 0.01 + 0.5 * (0.16 - 0.01) * (1 + np.cos(np.pi * np.arange(steps_e1) / (steps_e1 - 1)))
print("decay start/end:", round(eta_decay_e1[0], 3), round(eta_decay_e1[-1], 3))
assert round(eta_decay_e1[0], 3) == 0.16 and round(eta_decay_e1[-1], 3) == 0.01

▶ What you'll see: the cosine-decayed schedule starts high and ends low.

In [ ]:
theta_const_e1 = [0.0]
theta_decay_e1 = [0.0]
for i_e1 in range(steps_e1):
    theta_const_e1.append(theta_const_e1[-1] - eta_const_e1[i_e1] * 2 * (theta_const_e1[-1] - 3.0))
    theta_decay_e1.append(theta_decay_e1[-1] - eta_decay_e1[i_e1] * 2 * (theta_decay_e1[-1] - 3.0))
theta_const_e1 = np.array(theta_const_e1)
theta_decay_e1 = np.array(theta_decay_e1)
print("final theta:", round(theta_const_e1[-1], 3), round(theta_decay_e1[-1], 3))
assert abs(theta_const_e1[-1] - 3.0) < 0.01 and abs(theta_decay_e1[-1] - 3.0) < 0.01

▶ What you'll see: both schedules approach the optimum.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(theta_const_e1, label="constant", color="gray")
plt.plot(theta_decay_e1, label="cosine decay", color="purple")
plt.axhline(3.0, color="black", linestyle="--")
plt.title("Easy 1: optimization trajectory")
plt.xlabel("step")
plt.ylabel("θ")
plt.legend()
plt.show()

▶ What you'll see: the decayed schedule moves quickly early and then flattens near 3.

👀 Takeaway: schedules turn the same gradient rule into different training paths.

### Easy 2 — Combine warmup and cosine decay

**Goal.** Use warmup first and cosine decay afterward, because many deep-learning runs need both stable starts and smooth settling.

In [ ]:
warm_steps_e2 = 5
decay_steps_e2 = 25
eta_peak_e2 = 0.12
eta_min_e2 = 0.01
warm_e2 = eta_peak_e2 * (np.arange(1, warm_steps_e2 + 1) / warm_steps_e2)
t_e2 = np.arange(decay_steps_e2)
decay_e2 = eta_min_e2 + 0.5 * (eta_peak_e2 - eta_min_e2) * (1 + np.cos(np.pi * t_e2 / (decay_steps_e2 - 1)))
sched_e2 = np.concatenate([warm_e2, decay_e2[1:]])
print("length:", len(sched_e2), "peak:", round(sched_e2.max(), 3), "final:", round(sched_e2[-1], 3))
assert round(sched_e2.max(), 3) == 0.12 and round(sched_e2[-1], 3) == 0.01

▶ What you'll see: the schedule reaches 0.12 after warmup and ends at 0.01.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(sched_e2, marker="o", color="seagreen")
plt.axvline(warm_steps_e2 - 1, color="gray", linestyle="--", label="warmup ends")
plt.title("Easy 2: warmup + cosine")
plt.xlabel("step")
plt.ylabel("η_t")
plt.legend()
plt.show()

▶ What you'll see: an initial ramp followed by a smooth decay.

👀 Takeaway: warmup and cosine solve different parts of the training timeline.

### Easy 3 — Compare schedule budgets

**Goal.** Compare total learning-rate mass, because two schedules with the same maximum can spend very different amounts of movement.

In [ ]:
steps_e3 = 30
const_e3 = np.full(steps_e3, 0.06)
step_e3 = 0.12 * 0.5 ** (np.arange(steps_e3) // 10)
cos_e3 = 0.01 + 0.5 * (0.12 - 0.01) * (1 + np.cos(np.pi * np.arange(steps_e3) / (steps_e3 - 1)))
budgets_e3 = np.array([const_e3.sum(), step_e3.sum(), cos_e3.sum()])
print("budgets:", np.round(budgets_e3, 3))
assert np.all(budgets_e3 > 0)

▶ What you'll see: each schedule spends a different cumulative learning-rate budget.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["constant", "step", "cosine"], budgets_e3, color=["gray", "orange", "purple"])
plt.title("Easy 3: total η budget")
plt.ylabel("sum_t η_t")
plt.show()

▶ What you'll see: the highest budget belongs to the schedule that keeps rates larger for longer.

👀 Takeaway: matching peak learning rates does not match total movement.

### Easy 4 — Detect overshooting on a quadratic

**Goal.** Show that a too-large learning rate can bounce across the minimum, because update size can exceed the useful local distance.

In [ ]:
eta_good_e4 = 0.20
eta_bad_e4 = 1.10
steps_e4 = 10
theta_good_e4 = [0.0]
theta_bad_e4 = [0.0]
for step_e4 in range(steps_e4):
    theta_good_e4.append(theta_good_e4[-1] - eta_good_e4 * 2 * (theta_good_e4[-1] - 3.0))
    theta_bad_e4.append(theta_bad_e4[-1] - eta_bad_e4 * 2 * (theta_bad_e4[-1] - 3.0))
theta_good_e4 = np.array(theta_good_e4)
theta_bad_e4 = np.array(theta_bad_e4)
print("bad trajectory first values:", np.round(theta_bad_e4[:5], 3))
assert np.any(theta_bad_e4 > 3.0) and np.any(theta_bad_e4 < 0.0)

▶ What you'll see: the bad trajectory alternates around the optimum with growing magnitude.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(theta_good_e4, marker="o", label="η=0.20", color="teal")
plt.plot(theta_bad_e4, marker="x", label="η=1.10", color="crimson")
plt.axhline(3.0, color="black", linestyle="--")
plt.title("Easy 4: overshooting")
plt.xlabel("step")
plt.ylabel("θ")
plt.legend()
plt.show()

▶ What you'll see: the high-rate path crosses the target repeatedly instead of settling.

👀 Takeaway: schedules must respect the curvature of the loss surface.

### Easy 5 — Mini-batch noise and late small rates

**Goal.** Optimize with noisy gradients, because minibatches estimate the true gradient rather than measuring it exactly.

In [ ]:
steps_e5 = 60
rng_e5 = np.random.default_rng(5)
noise_e5 = rng_e5.normal(0, 0.5, size=steps_e5)
const_e5 = np.full(steps_e5, 0.08)
decay_e5 = 0.01 + 0.5 * (0.14 - 0.01) * (1 + np.cos(np.pi * np.arange(steps_e5) / (steps_e5 - 1)))
print("noise mean/std:", round(float(noise_e5.mean()), 3), round(float(noise_e5.std()), 3))
assert abs(noise_e5.mean()) < 0.2

▶ What you'll see: the synthetic minibatch noise is centered near zero.

In [ ]:
def noisy_run_e5(schedule_e5):
    theta_e5 = [0.0]
    for i_e5, eta_e5 in enumerate(schedule_e5):
        noisy_grad_e5 = 2 * (theta_e5[-1] - 3.0) + noise_e5[i_e5]
        theta_e5.append(theta_e5[-1] - eta_e5 * noisy_grad_e5)
    return np.array(theta_e5)

path_const_e5 = noisy_run_e5(const_e5)
path_decay_e5 = noisy_run_e5(decay_e5)
print("final errors:", round(abs(path_const_e5[-1] - 3), 3), round(abs(path_decay_e5[-1] - 3), 3))
assert abs(path_decay_e5[-1] - 3) < 0.2

▶ What you'll see: both runs are near the optimum, but late movement differs.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(path_const_e5, label="constant", color="gray")
plt.plot(path_decay_e5, label="cosine decay", color="purple")
plt.axhline(3.0, color="black", linestyle="--")
plt.title("Easy 5: noisy gradients")
plt.xlabel("step")
plt.ylabel("θ")
plt.legend()
plt.show()

▶ What you'll see: the decayed path jitters less late because its final rates are smaller.

👀 Takeaway: decaying rates reduce the effect of noisy gradient estimates near convergence.

## 🔴 Advanced

### Advanced 1 — Sweep maximum learning rate

**Goal.** Try several peak rates with the same cosine shape, because the schedule formula still needs a scale that matches the problem.

In [ ]:
peaks_a1 = np.array([0.04, 0.10, 0.30, 0.80])
steps_a1 = 45
final_losses_a1 = []
for peak_a1 in peaks_a1:
    sched_a1 = 0.005 + 0.5 * (peak_a1 - 0.005) * (1 + np.cos(np.pi * np.arange(steps_a1) / (steps_a1 - 1)))
    theta_a1 = 0.0
    for eta_a1 in sched_a1:
        theta_a1 = theta_a1 - eta_a1 * 2 * (theta_a1 - 3.0)
        theta_a1 = float(np.clip(theta_a1, -50, 50))
    final_losses_a1.append((theta_a1 - 3.0) ** 2)
final_losses_a1 = np.array(final_losses_a1)
print("final losses:", np.round(final_losses_a1, 5))
assert final_losses_a1[1] < final_losses_a1[0]

▶ What you'll see: too-low peaks can learn slowly, while reasonable peaks finish closer to the optimum.

In [ ]:
best_peak_a1 = peaks_a1[int(np.argmin(final_losses_a1))]
plt.figure(figsize=(5, 3))
plt.plot(peaks_a1, final_losses_a1, marker="o", color="navy")
plt.axvline(best_peak_a1, color="crimson", linestyle="--", label=f"best peak={best_peak_a1:.2f}")
plt.title("Advanced 1: peak-rate sweep")
plt.xlabel("η_max")
plt.ylabel("final loss")
plt.legend()
plt.show()

▶ What you'll see: one peak gives the smallest final loss for this toy problem.

👀 Takeaway: schedule shape and schedule scale are separate hyperparameters.

### Advanced 2 — One-cycle with momentum in the opposite direction

**Goal.** Pair a rising learning rate with falling momentum, because one-cycle training often trades stability from momentum against exploration from larger steps.

In [ ]:
up_a2 = np.linspace(0.02, 0.16, 8, endpoint=False)
down_a2 = np.linspace(0.16, 0.005, 22)
lr_a2 = np.concatenate([up_a2, down_a2])
mom_a2 = np.linspace(0.95, 0.85, len(up_a2)).tolist() + np.linspace(0.85, 0.95, len(down_a2)).tolist()
mom_a2 = np.array(mom_a2)
print("lr first/peak/final:", round(lr_a2[0], 3), round(lr_a2.max(), 3), round(lr_a2[-1], 3))
print("momentum first/min/final:", round(mom_a2[0], 3), round(mom_a2.min(), 3), round(mom_a2[-1], 3))
assert round(lr_a2.max(), 3) == 0.16 and round(mom_a2.min(), 3) == 0.85

▶ What you'll see: learning rate rises while momentum falls, then learning rate falls while momentum rises.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(lr_a2 / lr_a2.max(), label="scaled learning rate", color="crimson")
plt.plot((mom_a2 - mom_a2.min()) / (mom_a2.max() - mom_a2.min()), label="scaled momentum", color="teal")
plt.title("Advanced 2: one-cycle LR and momentum")
plt.xlabel("step")
plt.legend()
plt.show()

▶ What you'll see: the two curves move in opposite directions during the cycle.

👀 Takeaway: one-cycle can schedule both how far to step and how much velocity to carry.

### Advanced 3 — Simulate restarts with cosine cycles

**Goal.** Restart cosine decay repeatedly, because restarts periodically reintroduce larger exploratory steps.

In [ ]:
cycle_lengths_a3 = [8, 12, 16]
eta_max_a3 = 0.12
eta_min_a3 = 0.01
sched_parts_a3 = []
for length_a3 in cycle_lengths_a3:
    t_a3 = np.arange(length_a3)
    part_a3 = eta_min_a3 + 0.5 * (eta_max_a3 - eta_min_a3) * (1 + np.cos(np.pi * t_a3 / (length_a3 - 1)))
    sched_parts_a3.append(part_a3)
sched_a3 = np.concatenate(sched_parts_a3)
print("restart positions:", np.cumsum(cycle_lengths_a3)[:-1])
assert len(sched_a3) == sum(cycle_lengths_a3)

▶ What you'll see: the schedule is built from three cosine cycles.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(sched_a3, color="purple")
for pos_a3 in np.cumsum(cycle_lengths_a3)[:-1]:
    plt.axvline(pos_a3, color="gray", linestyle="--")
plt.title("Advanced 3: cosine restarts")
plt.xlabel("step")
plt.ylabel("η_t")
plt.show()

▶ What you'll see: each restart jumps back to a high learning rate before decaying again.

👀 Takeaway: restarts deliberately alternate settling phases with renewed exploration.

### Advanced 4 — Schedule weight decay strength through update size

**Goal.** Inspect how learning-rate decay also shrinks the effective L2 weight-decay update, because regularization steps are multiplied by $\eta_t$.

In [ ]:
weights_a4 = np.array([2.0, -1.0, 0.5])
lam_a4 = 0.1
sched_a4 = np.array([0.10, 0.05, 0.01])
decay_updates_a4 = np.array([eta_a4 * lam_a4 * weights_a4 for eta_a4 in sched_a4])
print("decay updates:\n", np.round(decay_updates_a4, 3))
assert np.allclose(decay_updates_a4[0], [0.02, -0.01, 0.005])

▶ What you'll see: the same weight vector receives smaller shrinkage as the learning rate decays.

In [ ]:
norms_a4 = np.linalg.norm(decay_updates_a4, axis=1)
plt.figure(figsize=(4.5, 3))
plt.bar(["η=.10", "η=.05", "η=.01"], norms_a4, color="darkorange")
plt.title("Advanced 4: effective decay update")
plt.ylabel("||ηλw||")
plt.show()

▶ What you'll see: the norm of the regularization movement falls with the learning rate.

👀 Takeaway: scheduled learning rates also schedule any update term multiplied by the rate.

### Advanced 5 — Choose a schedule with validation loss

**Goal.** Compare schedules on noisy train and validation curves, because the best schedule is the one that generalizes rather than merely moving fastest.

In [ ]:
steps_a5 = 50
rng_a5 = np.random.default_rng(20)
base_train_a5 = np.exp(-0.10 * np.arange(steps_a5))
base_val_a5 = 0.28 + 0.65 * np.exp(-0.08 * np.arange(steps_a5))
schedules_a5 = {
    "constant": np.full(steps_a5, 0.08),
    "step": 0.14 * 0.5 ** (np.arange(steps_a5) // 15),
    "cosine": 0.01 + 0.5 * (0.14 - 0.01) * (1 + np.cos(np.pi * np.arange(steps_a5) / (steps_a5 - 1))),
}
print("schedule names:", list(schedules_a5.keys()))
assert len(schedules_a5) == 3

▶ What you'll see: three candidate schedules will be evaluated.

In [ ]:
val_curves_a5 = {}
for name_a5, sched_a5 in schedules_a5.items():
    smooth_bonus_a5 = 0.08 * (sched_a5 / sched_a5.max())
    noise_a5 = rng_a5.normal(0, 0.01, size=steps_a5)
    val_curves_a5[name_a5] = base_val_a5 + smooth_bonus_a5 + noise_a5
final_vals_a5 = {name_a5: float(curve_a5[-1]) for name_a5, curve_a5 in val_curves_a5.items()}
best_name_a5 = min(final_vals_a5, key=final_vals_a5.get)
print("final validation losses:", {k: round(v, 3) for k, v in final_vals_a5.items()})
print("best schedule:", best_name_a5)
assert best_name_a5 in schedules_a5

▶ What you'll see: each schedule gets a final validation loss and one is selected.

In [ ]:
plt.figure(figsize=(5, 3))
for name_a5, curve_a5 in val_curves_a5.items():
    plt.plot(curve_a5, label=name_a5)
plt.title("Advanced 5: validation curves by schedule")
plt.xlabel("step")
plt.ylabel("validation loss")
plt.legend()
plt.show()

▶ What you'll see: the curves differ slightly; the lowest final validation curve wins.

👀 Takeaway: schedule choice is a validation decision, not just a prettier learning-rate plot.